<a href="https://colab.research.google.com/github/anelchik/anelsinternshipwork/blob/main/w06_validation_audit(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

**Lane:** Content Review Priority Ranking  
**Week-5 decision:** Which pseudonymized content pages should an SEO specialist review first?  
**Observed label:** `trend_direction == "down"`  

This audit applies the same standard to the FlyRank research claims and to my own Week-5 model: separate what was **observed** from what can be claimed, use a validation design that matches the decision, and check the final features for leakage.


## 1. Two paper findings + my methodology questions

I reviewed two findings from **FlyRank Research — The State of AI-Driven SEO in Numbers**.

### Finding A — Refreshing mature pages

The report's refresh analysis says that, for pages older than 180 days, **refreshed vs. stale pages show a median impression effect of +588 with a 95% CI of [548, 634]**. It also reports that 7 of 9 refresh strata have a statistically significant lift.

**Where the outcome comes from:** refresh status is based on whether an older page was updated, and the measured outcome is the change/lift in search impressions.

**Methodology question:** *Were refreshed and stale pages comparable before the refresh decision?* A team may preferentially refresh pages that already have more traffic, stronger historical visibility, or higher business value. A held-out significance test helps with estimation, but it does not by itself remove selection bias. I would want a matched/stratified comparison (including client and prior visibility) or a controlled pre/post design before reading the number as a causal effect of refreshing.

### Finding B — Which pages will grow?

The report says its growth model was trained on **96.6K clearly growing or declining pages**, reaching about **90% accuracy on new pages from the same brands** and **75% on brands it had not seen before**.

**Where the label comes from:** the paper defines trend direction from the last 30 days versus the previous 30 days: `up` is more than +10%, `down` is more than -10%, with a stable band around zero.

**Methodology question:** *Does the validation design support the forward-looking phrase “will grow”?* Holding out brands is useful because it measures transfer to unseen brands, but a future-growth claim is strongest when the test period is later in time than the training period. If features and labels come from the same recent window, a grouped split can still be honest cross-brand validation while not being a full time-forward forecast test.

**Constructive takeaway:** both findings are useful evidence. My questions are about the boundary of the claim, not about dismissing the measured results.


In [ ]:
import pandas as pd
from IPython.display import display

paper_audit = pd.DataFrame([
    {
        "finding": "Refresh old pages",
        "reported_result": "180+ day refreshed vs stale: median +588 impressions; 95% CI [548, 634]",
        "validation_question": "Are refreshed and stale pages comparable before treatment, including prior visibility/client?",
    },
    {
        "finding": "Growth prediction",
        "reported_result": "~90% same-brand; ~75% unseen-brand accuracy",
        "validation_question": "Is there a later-time holdout that supports the forward-looking 'will grow' wording?",
    },
])

display(paper_audit)


,finding,reported_result,validation_question
0,Refresh old pages,180+ day refreshed vs stale: median +588 impre...,Are refreshed and stale pages comparable befor...
1,Growth prediction,~90% same-brand; ~75% unseen-brand accuracy,Is there a later-time holdout that supports th...


## 2. My model under an honest split (before/after)

My Week-5 notebook **already used a 75/25 grouped holdout by `client_id`**, which is the honest design I want to keep. For this audit I add a deliberately naïve **random row split** as the “before” diagnostic, then rerun the same Random Forest under the grouped split as the “after”.

Why this matters: pages from the same client can share content, tracking, and site-level patterns. A random row split allows the same client to appear in both train and test, so it can make transfer performance look better than it really is.

I keep the model, features, random seed, and 25% test size the same. The only important change is the split rule.


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import sklearn
from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)
RANDOM_STATE = 42

CANDIDATE_PATHS = [
    Path("content_refresh_anonymized.csv"),
    Path("content_refresh_anonymized(1).csv"),
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
    Path("/mnt/data/content_refresh_anonymized.csv"),
    Path("/mnt/data/content_refresh_anonymized(1).csv"),
]

data_path = next((p for p in CANDIDATE_PATHS if p.exists()), None)
if data_path is None:
    raise FileNotFoundError("Could not find content_refresh_anonymized.csv")

df = pd.read_csv(data_path)
y = (df["trend_direction"] == "down").astype(int)
groups = df["client_id"]

forbidden_features = {
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "impressions_last_30d",
    "impressions_prev_30d",
}
feature_columns = [c for c in df.columns if c not in forbidden_features]
X = df[feature_columns].copy()

print(f"Loaded {df.shape[0]:,} rows and {df.shape[1]} columns")
print(f"Observed 'down' rate: {y.mean():.3f}")
print(f"scikit-learn: {sklearn.__version__}")


def make_model(X_train):
    numeric_features = X_train.select_dtypes(exclude=["object", "category"]).columns.tolist()
    categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

    preprocessor = ColumnTransformer([
        ("numeric", Pipeline([
            ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
            ("scaler", StandardScaler()),
        ]), numeric_features),
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), categorical_features),
    ])

    estimator = RandomForestClassifier(
        n_estimators=250,
        min_samples_leaf=8,
        max_features="sqrt",
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )
    return Pipeline([("preprocessor", preprocessor), ("model", estimator)])


def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))[: min(k, len(y_true))]
    return float(np.asarray(y_true)[order].mean())


def fit_and_score(name, train_idx, test_idx):
    model = make_model(X.iloc[train_idx])
    model.fit(X.iloc[train_idx], y.iloc[train_idx])
    scores = model.predict_proba(X.iloc[test_idx])[:, 1]
    return model, scores, {
        "split": name,
        "test_rows": len(test_idx),
        "test_positive_rate": y.iloc[test_idx].mean(),
        "average_precision": average_precision_score(y.iloc[test_idx], scores),
        "roc_auc": roc_auc_score(y.iloc[test_idx], scores),
        "precision_at_20": precision_at_k(y.iloc[test_idx], scores, 20),
        "precision_at_50": precision_at_k(y.iloc[test_idx], scores, 50),
    }

# BEFORE diagnostic: random rows. This is not my preferred validation design.
all_idx = np.arange(len(df))
random_train_idx, random_test_idx = train_test_split(
    all_idx,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y,
)
random_model, random_scores, random_result = fit_and_score(
    "Before: random row split", random_train_idx, random_test_idx
)

# AFTER / honest Week-5 design: whole clients held out.
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE)
group_train_idx, group_test_idx = next(gss.split(X, y, groups=groups))

group_model, group_scores, group_result = fit_and_score(
    "After: grouped by client", group_train_idx, group_test_idx
)

train_clients = set(groups.iloc[group_train_idx])
test_clients = set(groups.iloc[group_test_idx])
client_overlap = len(train_clients & test_clients)
assert client_overlap == 0

comparison = pd.DataFrame([random_result, group_result])
display(comparison.round(3))
print("Grouped split client overlap:", client_overlap)
print(
    "AP change (grouped - random): "
    f"{group_result['average_precision'] - random_result['average_precision']:+.3f}"
)

# Concrete failure examples from the honest grouped test split.
review = df.iloc[group_test_idx][
    ["content_id", "content_type", "impressions_90d", "ctr", "avg_position"]
].copy()
review["actual_down"] = y.iloc[group_test_idx].to_numpy()
review["model_score"] = group_scores
review["rank"] = review["model_score"].rank(method="first", ascending=False).astype(int)

false_positives = review[
    (review["rank"] <= 50) & (review["actual_down"] == 0)
].sort_values("rank").head(3)

missed_declines = review[
    review["actual_down"] == 1
].sort_values("model_score").head(3)

print("\nThree false positives in the top-50 review queue:")
display(false_positives.round(4))
print("Three observed declines assigned very low scores:")
display(missed_declines.round(4))


Loaded 30,000 rows and 44 columns
Observed 'down' rate: 0.542
scikit-learn: 1.8.0


,split,test_rows,test_positive_rate,average_precision,roc_auc,precision_at_20,precision_at_50
0,Before: random row split,7500,0.542,0.806,0.793,1.0,0.96
1,After: grouped by client,7115,0.517,0.608,0.623,0.6,0.62


Grouped split client overlap: 0
AP change (grouped - random): -0.198

Three false positives in the top-50 review queue:


,content_id,content_type,impressions_90d,ctr,avg_position,actual_down,model_score,rank
22042,content_2ba626fea4d6,keyword article,360,0.00,7.2,0,0.9416,3
29456,content_b46c62b14582,keyword article,6240,0.13,31.8,0,0.9100,7
4050,content_500bd3907331,keyword article,4037,0.10,5.5,0,0.9090,8


Three observed declines assigned very low scores:


,content_id,content_type,impressions_90d,ctr,avg_position,actual_down,model_score,rank
1864,content_16f38acf0f26,keyword article,2,0.00,50.0,1,0.1374,7065
1724,content_b72b06434946,keyword article,1061,0.19,41.0,1,0.1400,7063
27271,content_7bc32bc1df59,keyword article,1,0.00,0.0,1,0.1446,7057


## 3. Leakage audit

The target is an observed trend label, so anything that directly encodes the same last-30-days-versus-previous-30-days calculation is not a legitimate predictor.

My final feature set therefore excludes:

- `trend_direction` — the target itself;
- `trend_pct` — the direct numeric source of the label;
- `impressions_last_30d` and `impressions_prev_30d` — together they reconstruct `trend_pct` almost exactly;
- `content_id` and `client_id` — identifiers, not model features (`client_id` is used only to define groups).

The code below checks both the exclusion list and the reconstruction risk. A near-perfect reconstruction of `trend_pct` is strong evidence that those two recent impression windows would leak the answer.


In [ ]:
# 1) Confirm forbidden fields are not in the model matrix.
leaked_into_X = sorted(set(X.columns) & forbidden_features)
print("Forbidden fields present in X:", leaked_into_X)
assert leaked_into_X == []

# 2) Demonstrate why the two recent impression windows are forbidden.
prev = df["impressions_prev_30d"].replace(0, np.nan)
reconstructed_trend_pct = (
    (df["impressions_last_30d"] - df["impressions_prev_30d"]) / prev
) * 100

valid = (
    reconstructed_trend_pct.notna()
    & df["trend_pct"].notna()
    & np.isfinite(reconstructed_trend_pct)
)

reconstruction_corr = np.corrcoef(
    reconstructed_trend_pct[valid], df.loc[valid, "trend_pct"]
)[0, 1]
mean_abs_error = np.mean(
    np.abs(reconstructed_trend_pct[valid] - df.loc[valid, "trend_pct"])
)

leakage_summary = pd.DataFrame({
    "check": [
        "Recent impression windows vs trend_pct",
        "Forbidden fields inside final X",
        "Grouped train/test client overlap",
    ],
    "result": [
        f"corr={reconstruction_corr:.9f}; mean abs error={mean_abs_error:.4f}",
        str(leaked_into_X),
        str(client_overlap),
    ],
    "decision": [
        "Exclude both recent impression-window fields",
        "Pass",
        "Pass",
    ],
})

display(leakage_summary)


Forbidden fields present in X: []


,check,result,decision
0,Recent impression windows vs trend_pct,corr=0.999999998; mean abs error=0.0220,Exclude both recent impression-window fields
1,Forbidden fields inside final X,[],Pass
2,Grouped train/test client overlap,0,Pass


## 4. Claim rewrite

### Too strong

> “The model can identify which pages are going to decline.”

That sentence turns an association model into a forecast and hides the generalization drop revealed by the grouped split.

### Safe version

> **On a held-out client-grouped split, the Random Forest achieved measured average precision of about 0.61 for ranking pages associated with an observed decline. The score is directional decision-support for prioritizing human review; it does not diagnose the cause of decline, guarantee a future decline, or prove that a specific intervention will improve the page.**

### What the before/after audit changes

The random row split produces much stronger metrics than the grouped split (about **0.81 AP vs. 0.61 AP** in this run). That gap is evidence that validation design materially changes the apparent strength of the model. I therefore report the grouped result as the honest headline and keep the random result only as an audit demonstration.

The failure examples above reinforce the same limit: some high-scored pages are not labeled down, and some observed declines receive low scores. The model is a review-priority signal, not an automatic decision-maker.


In [ ]:
claim_audit = pd.DataFrame([
    {
        "claim_part": "Outcome",
        "safe_language": "associated with an observed decline",
        "why": "The label is retrospective; it is not a causal diagnosis.",
    },
    {
        "claim_part": "Evidence",
        "safe_language": "measured on a held-out client-grouped split",
        "why": "Whole clients are unseen at evaluation time.",
    },
    {
        "claim_part": "Use",
        "safe_language": "directional decision-support for human review",
        "why": "Errors remain in both directions, so the score should prioritize rather than automate.",
    },
])

display(claim_audit)


,claim_part,safe_language,why
0,Outcome,associated with an observed decline,The label is retrospective; it is not a causal...
1,Evidence,measured on a held-out client-grouped split,Whole clients are unseen at evaluation time.
2,Use,directional decision-support for human review,"Errors remain in both directions, so the score..."


## Self-check

Before submission:

- [x] Every section above is filled — markdown thinking **and** executable code
- [x] The notebook runs top to bottom with no errors
- [x] No client names, URLs, or private queries are displayed
- [x] Claims use careful language: **observed, measured, directional, decision-support**
- [x] The honest split holds out whole clients and confirms zero client overlap
- [x] Leakage fields are explicitly excluded and the reconstruction risk is demonstrated
- [x] Concrete false-positive and missed-decline examples are shown
- [ ] Commit this notebook as `work/notebooks/w06_validation_audit.ipynb`, then submit the repository URL on the assignment card
